# XGBoost: predictive performance

Predict two-year recidivism (`Two_yr_Recidivism`) on the COMPAS cohort with XGBoost. Training lives in `xgb_model.py` (run it once to save the model to `models/`); the metrics live in `performance.py`. This notebook loads the saved model and shows the results.

- Shared stratified 70/30 train/test split (`compas_scoring.data.train_test`), so results are comparable with the other models.
- Hyperparameters picked by a randomized search with 5-fold CV on the **training set only**.
- The test set is used once, to report performance next to the incumbent COMPAS score.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys

sys.path.insert(0, "../src")

import matplotlib.pyplot as plt

from performance import confusion, performance_table, roc_data, scores
from xgb_model import load_data, load_metadata, load_model

## 1. Data

Feature set FS1 (`race_aware`): all 10 predictors. Change `FEATURE_SET` to `"selected"` (FS2) or `"race_blind"` (FS3) to rerun on the other sets.

In [ ]:
FEATURE_SET = "race_aware"

train, test = load_data(FEATURE_SET)
print(f"{len(train)} train rows, {len(test)} test rows")
print(f"Recidivism rate - train: {train.base_rate:.3f}, test: {test.base_rate:.3f}")
train.X.head()

## 2. Saved model

Trained and tuned by `python xgboost/xgb_model.py`.

In [ ]:
model = load_model(FEATURE_SET)
meta = load_metadata(FEATURE_SET)

print(f"Best CV AUC: {meta['cv_auc']:.4f}")
meta["best_params"]

## 3. Test-set performance

Threshold 0.5. COMPAS is only available as a high/low label (`score_factor`), so its AUC and Brier score are those of a single cut-off.

In [ ]:
performance_table(model, test)

In [ ]:
confusion(model, test)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
for name, y_score in [("XGBoost", scores(model, test)), ("COMPAS (incumbent)", test.incumbent)]:
    roc = roc_data(test.y, y_score)
    ax.plot(roc.fpr, roc.tpr, label=name)
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set(xlabel="False positive rate", ylabel="True positive rate", title="ROC curve - test set")
ax.legend()
plt.show()